# Diagnose v6 output vs IFP2 reference

Goal: figure out WHY 192 of 213 rows came back as 'Not Applicable'.

Compares:
- `output_benefits_212.json` (what we produced)
- IFP2 reference CSV for the same plan(s)
- Original input CSV (so we can see what the LLM was actually shown)

Outputs a classification of every wrong row into one of these buckets:
- INPUT_HAS_NO_DATA - the input genuinely had no rows for this benefit
- ROWS_TOO_FEW - input has 1-2 rows, LLM couldn't tell what to make
- ROWS_PRESENT_LLM_BAILED - input has cost data but LLM said 'Not Applicable'
- PHANTOM_OON - we created an OutOfNetwork row with no OON input data


## 0. Config

In [ ]:
import json
from pathlib import Path
import pandas as pd

WORK_DIR = Path.cwd()

# ----- INPUTS -----
OUR_OUTPUT = WORK_DIR / 'output_benefits_212.json'
ORIGINAL_INPUT = WORK_DIR / 'input_MedicareFeed_carrierDESR_PBP_LoadID211.csv'  # update if 212 has its own file

# Reference IFP2 CSV - the ground truth. Adjust to whichever you have.
IFP2_REF = WORK_DIR / 'output_ifp2_planbenefits_planID_H0107_003_endingPeriodID_336.csv'

# Which plan to diagnose
TARGET_PLAN_ID = 'HCSC_H0107_003_0'

print(f'Our output:    {OUR_OUTPUT}')
print(f'Original input: {ORIGINAL_INPUT}')
print(f'IFP2 reference: {IFP2_REF}')
print(f'Target plan:    {TARGET_PLAN_ID}')


## 1. Load all three sources

In [ ]:
# Our v6 output
with open(OUR_OUTPUT) as f:
    our_envelope = json.load(f)
our_rows = pd.DataFrame(our_envelope['results'])
our_plan = our_rows[our_rows['planid'] == TARGET_PLAN_ID].copy()
print(f'Our output: {len(our_rows)} total rows | {len(our_plan)} for {TARGET_PLAN_ID}')
print(f'  load_id in envelope: {our_envelope.get("load_id")!r}')

# IFP2 reference
ref = pd.read_csv(IFP2_REF, engine='python', on_bad_lines='skip')
# Reference uses 'benefitID', 'description', etc - we use 'benefitid', 'benefitdesc'
print(f'\nIFP2 reference: {len(ref)} rows for plan {ref["planID"].iloc[0] if not ref.empty else "?"}')
print(f'  Columns: {list(ref.columns)}')

# Original input
inp = pd.read_csv(ORIGINAL_INPUT).dropna(subset=['FileName'])
input_plan_fn = TARGET_PLAN_ID.replace('_', '-').replace('HCSC-', 'HCSC_', 1)
inp_plan = inp[inp['FileName'] == 'HCSC_H0107-003-000'].copy()
print(f'\nOriginal input: {len(inp)} total rows | {len(inp_plan)} for HCSC_H0107-003-000')


## 2. Identify the 'Not Applicable' rows in our output

In [ ]:
# Find rows that came back as Not Applicable / Not Covered / See Brochure
def is_bogus(s):
    if not isinstance(s, str):
        return True
    s_low = s.strip().lower()
    bogus_phrases = ['not applicable', 'not covered', 'see brochure', 'n/a', 'not available']
    return any(p in s_low for p in bogus_phrases) or s_low == ''

our_plan['is_bogus'] = our_plan['benefitdesc'].apply(is_bogus)
bogus_rows = our_plan[our_plan['is_bogus']].copy()
real_rows = our_plan[~our_plan['is_bogus']].copy()

print(f'For {TARGET_PLAN_ID}:')
print(f'  Total rows we produced: {len(our_plan)}')
print(f'  Real content rows:      {len(real_rows)}  ({100*len(real_rows)/len(our_plan):.0f}%)')
print(f'  Bogus content rows:     {len(bogus_rows)}  ({100*len(bogus_rows)/len(our_plan):.0f}%)')
print()
print('Sample real rows:')
for _, r in real_rows.head(5).iterrows():
    print(f'  bid={r["benefitid"]:>4} ctID={r["coverageTypeid"]} desc={r["benefitdesc"][:60]}')
print()
print('Sample bogus rows:')
for _, r in bogus_rows.head(5).iterrows():
    print(f'  bid={r["benefitid"]:>4} ctID={r["coverageTypeid"]} desc={r["benefitdesc"][:60]}')


## 3. What does the reference say for each bogus row?

In [ ]:
# For each bogus row in our output, look up what the reference has
ref_by_bid_ct = {}
for _, r in ref.iterrows():
    key = (int(r['benefitID']), int(r['coverageTypeID']))
    ref_by_bid_ct.setdefault(key, []).append({
        'description': r.get('description', ''),
        'amount': r.get('amount', None),
        'serviceTypeID': r.get('serviceTypeID', 0),
    })

# Categorize each bogus row
buckets = {'INPUT_HAS_NO_DATA': [], 'REF_HAS_REAL_DATA': [], 'PHANTOM_OON': [], 'NO_REF_ROW': []}
for _, r in bogus_rows.iterrows():
    bid, ct = int(r['benefitid']), int(r['coverageTypeid'])
    ref_match = ref_by_bid_ct.get((bid, ct), [])

    if not ref_match:
        buckets['NO_REF_ROW'].append((bid, ct, r['benefitname']))
        continue

    # Is the reference also bogus?
    ref_descs = [m['description'] for m in ref_match]
    ref_all_bogus = all(is_bogus(d) for d in ref_descs)
    if ref_all_bogus:
        buckets['INPUT_HAS_NO_DATA'].append((bid, ct, r['benefitname']))
    else:
        sample_real_desc = next(d for d in ref_descs if not is_bogus(d))
        buckets['REF_HAS_REAL_DATA'].append((bid, ct, r['benefitname'], sample_real_desc))

print(f'Classification of {len(bogus_rows)} bogus rows:')
for name, items in buckets.items():
    print(f'  {name}: {len(items)}')

# The most important bucket - rows where reference has REAL data and we said "Not Applicable"
print()
print('=== ROWS WHERE WE FAILED BUT REFERENCE HAS REAL DATA ===')
print(f'(These are the most fixable cases - input likely had cost data)')
for item in buckets['REF_HAS_REAL_DATA'][:20]:
    bid, ct, bname, ref_desc = item
    print(f'  bid={bid:>4} ct={ct} {bname[:30]:<30} REF: {str(ref_desc)[:60]}')


## 4. For one specific 'wrong' row, dump the actual input we showed the LLM

In [ ]:
# Pick a specific benefit ID where we said 'Not Applicable' but reference has real data
# Default to the first one
if buckets['REF_HAS_REAL_DATA']:
    INSPECT_BID, INSPECT_CT, INSPECT_NAME, INSPECT_REF_DESC = buckets['REF_HAS_REAL_DATA'][0]
else:
    INSPECT_BID, INSPECT_CT = 900, 1
    print('No REF_HAS_REAL_DATA cases - this is good! Or maybe the diagnosis is wrong.')

print(f'Inspecting bid={INSPECT_BID} ct={INSPECT_CT} ({INSPECT_NAME})')
print(f'  Reference says:  {INSPECT_REF_DESC}')
print(f'  We produced:     {bogus_rows[(bogus_rows["benefitid"] == INSPECT_BID) & (bogus_rows["coverageTypeid"] == INSPECT_CT)]["benefitdesc"].iloc[0]}')
print()

# What input rows did the LLM see for this benefit?
# We need to replicate the group_rows_by_benefit logic from build_benefits_v6 here
import sys, re
sys.path.insert(0, str(WORK_DIR))
from build_benefits_v6 import SECTION_CODE_TO_BENEFIT_ID, PLAN_LEVEL_CATEGORIES, _extract_section_code, _is_oon

is_oon_target = (INSPECT_CT == 2)

# Get all rows for this plan whose category maps to this benefit ID
matched_rows = []
for _, row in inp_plan.iterrows():
    cat = row['category'] if isinstance(row['category'], str) else ''
    sc = _extract_section_code(cat)
    oon = _is_oon(cat)
    bid = None
    if sc and sc in SECTION_CODE_TO_BENEFIT_ID:
        bid = SECTION_CODE_TO_BENEFIT_ID[sc]
    elif cat in PLAN_LEVEL_CATEGORIES:
        bid = PLAN_LEVEL_CATEGORIES[cat]
    else:
        for prefix, pbid in PLAN_LEVEL_CATEGORIES.items():
            if cat.startswith(prefix + '.'):
                bid = pbid
                break
    if bid == INSPECT_BID and oon == is_oon_target:
        matched_rows.append(row)

print(f'Input rows that mapped to bid={INSPECT_BID} (is_oon={is_oon_target}):')
print(f'  Count: {len(matched_rows)}')
if matched_rows:
    print()
    for r in matched_rows[:20]:
        print(f'  [{r["category"][:50]:<50}] {str(r["field"])[:30]:<30} = {str(r["value"])[:40]}')
else:
    print(f'  NO INPUT ROWS - explains why LLM said "Not Applicable"')
    print(f'  Likely SECTION_CODE_TO_BENEFIT_ID is missing the section code for this benefit')


## 5. Verdict and recommendation

In [ ]:
print('=' * 70)
print('DIAGNOSTIC SUMMARY')
print('=' * 70)

ref_data_count = len(buckets['REF_HAS_REAL_DATA'])
no_input_count = len(buckets['INPUT_HAS_NO_DATA'])
phantom_count = len(buckets['NO_REF_ROW'])

print(f'\nOf {len(bogus_rows)} bogus rows in our output:')
print(f'  {ref_data_count} have REAL data in reference but we said Not Applicable')
print(f'    -> Either input rows are being grouped to the wrong benefit_id,')
print(f'       OR the section-code map is wrong,')
print(f'       OR the LLM is bailing on cases with sparse rows.')
print(f'')
print(f'  {phantom_count} have no corresponding row in the reference at all')
print(f'    -> We are creating phantom benefits/coverage combinations that')
print(f'       should not exist. Probably the OON-variant logic is too aggressive.')
print(f'')
print(f'  {no_input_count} have no input data and reference is also bogus')
print(f'    -> These are fine; reference also has them as Not Applicable.')

print()
print('=' * 70)
print('NEXT STEPS')
print('=' * 70)
if phantom_count > 50:
    print('Phantom rows are the #1 problem.')
    print('Fix: stop creating OON variants of benefits when no OON input rows exist.')
elif ref_data_count > 50:
    print('Wrong input grouping is the #1 problem.')
    print('Fix: re-examine section-code mappings and check which categories map to which benefits.')
else:
    print('Mixed problems - share this output with Claude for targeted fixes.')
